In [3]:
import tensorflow as tf
import os

print("TF version:", tf.__version__)
print("GPU:", tf.config.list_physical_devices('GPU'))

base = '/kaggle/input/datasets/masoudnickparvar/brain-tumor-mri-dataset'
print("\nDataset folders:")
for f in os.listdir(base):
    print(" ", f)

TF version: 2.19.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU'), PhysicalDevice(name='/physical_device:GPU:1', device_type='GPU')]

Dataset folders:
  Training
  Testing


In [7]:
!cd brain-tumor-image-classification && git pull
!python brain-tumor-image-classification/training/train.py

remote: Enumerating objects: 42, done.
remote: Counting objects: 100% (42/42), done.
remote: Compressing objects: 100% (32/32), done.
remote: Total 37 (delta 4), reused 36 (delta 3), pack-reused 0 (from 0)
Unpacking objects: 100% (37/37), 69.73 KiB | 2.32 MiB/s, done.
From https://github.com/prem332/brain-tumor-image-classification
   b6e8a13..15a0e7e  main       -> origin/main
Updating b6e8a13..15a0e7e
Fast-forward
 cloudbuild.yaml                       |   71 +-
 frontend/.dockerignore                |    6 +
 frontend/.gitignore                   |   41 +
 frontend/.gitkeep                     |    0
 frontend/AGENTS.md                    |    5 +
 frontend/CLAUDE.md                    |    1 +
 frontend/Dockerfile                   |   36 +
 frontend/README.md                    |   36 +
 frontend/app/favicon.ico              |  Bin 0 -> 25931 bytes
 frontend/app/globals.css              |   26 +
 frontend/app/layout.tsx               |   33 +
 frontend/app/page.tsx                

In [11]:
import os
import json
from kaggle_secrets import UserSecretsClient
from google.cloud import storage

secrets = UserSecretsClient()
sa_key  = secrets.get_secret("GCP_SA_KEY")

with open('/kaggle/working/gcp_key.json', 'w') as f:
    f.write(sa_key)

os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = '/kaggle/working/gcp_key.json'

client = storage.Client(project='brain-tumor-ai-prod')
bucket = client.bucket('brain-tumor-ai-models')

files = [
    ('/kaggle/working/vgg_model_v3.h5',      'models/v3/vgg_model.h5'),
    ('/kaggle/working/model_config.json',     'models/v3/model_config.json'),
    ('/kaggle/working/training_curves.png',   'models/v3/training_curves.png'),
    ('/kaggle/working/confusion_matrix.png',  'models/v3/confusion_matrix.png'),
]

print("Uploading to GCS...")
for local, gcs_path in files:
    blob = bucket.blob(gcs_path)
    blob.upload_from_filename(local)
    print(f"  ✓ gs://brain-tumor-ai-models/{gcs_path}")

os.remove('/kaggle/working/gcp_key.json')
print("\nKey deleted")
print("All uploads complete!")

Uploading to GCS...
  ✓ gs://brain-tumor-ai-models/models/v3/vgg_model.h5
  ✓ gs://brain-tumor-ai-models/models/v3/model_config.json
  ✓ gs://brain-tumor-ai-models/models/v3/training_curves.png
  ✓ gs://brain-tumor-ai-models/models/v3/confusion_matrix.png

Key deleted
All uploads complete!
